In [1]:
import sys

from brian2 import Hz
import numpy as np 
import pandas as pd 
import os 
import pickle

from brian2 import Hz,mV
from pathlib import Path
REPO_ROOT = Path.cwd().resolve().parents[4]
sys.path.insert(0, str(REPO_ROOT / "code"))
from model_unified_ver import run_exp
from model_unified_ver import default_params as params
import utils as utl
from analysis_of_simulation_result import *
from fitting import hill
from make_network import *
av1a1 = [720575940623041549,720575940622894616,720575940626958878,720575940633984924,720575940611137742,720575940627192337]


av1a1 = [720575940623041549,720575940622894616,720575940626958878,720575940633984924,720575940611137742,720575940627192337]
TPN1 = [720575940623118029, 720575940624967561]

atGRN_cluster = pd.read_parquet(REPO_ROOT/'data/atGRN_cluster_info_v783.parquet')
atgrn_c2g = {}
for t in np.unique(atGRN_cluster.type):
    atgrn_c2g[t] = list(atGRN_cluster.flyid.values[atGRN_cluster.type==t])


In [11]:
# av1a1 
cell_in_network_per_type = {}
rank_df = pd.read_parquet('/volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/LOF_simulation_with_fixed_firing_pattern_final-base_PER/av1a1_labial/labial_av1a1_rank.parquet')
in_network_av1a1_labial = rank_df[np.any(rank_df<=20,axis=1)].index.values.astype(int)
in_network_av1a1_labial_cell = target_ids_valid_all[np.isin(g_info_all,in_network_av1a1_labial)]

# av1a1 
rank_df = pd.read_parquet('/volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/LOF_simulation_with_fixed_firing_pattern_final-base_PER/av1a1_tarsal/tarsal_av1a1_rank.parquet')
in_network_av1a1_tarsal = rank_df[np.any(rank_df<=20,axis=1)].index.values.astype(int)
in_network_av1a1_tarsal_cell = target_ids_valid_all[np.isin(g_info_all,in_network_av1a1_tarsal)]


inhibitory_neurons1 = [c for c in in_network_av1a1_labial_cell if np.sum(syn_df[syn_df.Presynaptic_ID==c]['Excitatory']<0)>0]#cell_in_network_per_type['av1a1']
inhibitory_type_labial = list(np.unique([fid2g[x] for x in inhibitory_neurons1]).astype(int))
inhibitory_neurons2 = [c for c in in_network_av1a1_tarsal_cell if np.sum(syn_df[syn_df.Presynaptic_ID==c]['Excitatory']<0)>0]#cell_in_network_per_type['av1a1']
inhibitory_type_tarsal = list(np.unique([fid2g[x] for x in inhibitory_neurons2]).astype(int))


inhibitory_type = np.union1d(inhibitory_type_labial,inhibitory_type_tarsal)
inhibitory = inhibitory_type
inhibitory = {}
inhibitory['atGRN+TPN1'] = inhibitory_neurons2
inhibitory['L1+L2+L3'] = inhibitory_neurons1

In [12]:
params_opt_tarsal = pickle.load(open(REPO_ROOT/'figure4/figure4F-K.MN9_PER_fitting/fitting_result/tarsal/result_av1a1_170.pkl','rb'))

V_at,K_at,n_at,V_tp,K_tp,n_tp,k_act_T,r0_T,h_T = params_opt_tarsal.x


firing = [(0,0),*[(np.round(hill(c,V_at,K_at,n_at),1),np.round(hill(c,V_tp,K_tp,n_tp),1)) for c in [10,50,100,500]]]

[(0, 0), (33.1, 62.1), (69.5, 76.2), (88.4, 82.5), (127.2, 96.7)]


In [13]:
interest_neurons = np.union1d(inhibitory['L1+L2+L3'],inhibitory['atGRN+TPN1'])
interest_neurons_2_id = {i:j for j,i in enumerate(interest_neurons)}
# numiter = 0 
# spike_data_per_trial = []
# for i,t_df in spikes[numiter].groupby(by='trial'):
#     spike_data = {}
#     temp = {}
#     for j,spike_t_df in t_df.groupby(by='flywire_id'):
#         temp[interest_neurons_2_id[j]] = spike_t_df.t.values
#     for ii in range(len(interest_neurons)):
#         if ii not in temp.keys():
#             temp[ii] = np.array([])
#     temp = dict(sorted(temp.items()))
#     spike_data[1] = []
#     spike_data[2] = []
#     spike_data[3] = list(temp.values())
#     spike_data_per_trial.append(spike_data)
# with open(f'/volume_4/research/seongbong/flywire/geosmin_project_version_update/figure5/susceptibility_difference/3. Redundancy/geosmin_effect_activation_of_same_set_of_interneurons/inhibitory_spike_data/{0}Hz_{0}Hz_170Hz.pkl','wb') as f:
#     pickle.dump(spike_data_per_trial,f)
    

In [14]:
for ff1,ff2 in firing[1:]:
    for p in np.arange(0,1,0.05):
        percent = f'{np.round(p*100,2)}%'

        params['w_syn'] = 0.275*mV
        params['n_run'] = 100
        config = {
            'path_res'  : f'{os.getcwd()}/result/{percent}',                              # directory to store results
           'path_comp' : REPO_ROOT / "data" /'Completeness_783.csv',    
            'path_con'  : f'{os.getcwd()}/data/synapse_edges_rm_labial_{percent}.parquet',    # connectivity data
            'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
        }
        if percent not in os.listdir(f'{os.getcwd()}/result'):
            os.mkdir(config['path_res'])
        if 'input_spike_pattern' not in os.listdir(f'{os.getcwd()}/result/{percent}'):
            os.mkdir(f"{config['path_res']}/input_spike_pattern")
        if 'params' not in os.listdir(f'{config["path_res"]}'):
            os.mkdir(f'{config["path_res"]}/params')

        pickle.dump(params,open(f'{config["path_res"]}/params/params.pkl','wb'))


        neu_exc = atgrn_c2g['a6']+atgrn_c2g['a7']

        neu_add = [TPN1,interest_neurons]
        except_output_ids = [atgrn_c2g['a6']+atgrn_c2g['a7']+TPN1]
        except_output_wsyn = [0.825*mV]
        except_input_ids = [interest_neurons]
        except_input_wsyn = [0*mV]
        neu_slnc = []

        spike_path = REPO_ROOT/f'figure4/concentration_mapped_simulation/result_atGRN&TPN1/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'
        predetermined_input_or_not = [1,1,0]
        params['r_poi1'] = ff1 * Hz
        params['r_poi2'] = ff2 * Hz
        params['r_poi3'] = 0 * Hz
        run_exp(exp_name=f'{ff1}Hz_{ff2}Hz_0Hz', neu_exc=neu_exc,neu_exc_add=neu_add,Except_output_Ids=except_output_ids,Except_output_w_syn=except_output_wsyn,Except_input_Ids=except_input_ids,Except_input_w_syn=except_input_wsyn,neu_slnc=neu_slnc,params=params,predetermined_input_or_not=predetermined_input_or_not,spike_path=spike_path, **config)

>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/0.0%/33.1Hz_62.1Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   59 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/5.0%/33.1Hz_62.1Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   51 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/10.0%/33.1Hz_62.1Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   52 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/fig

WARNING    /home/se/anaconda3/envs/flywire/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]


    Elapsed time:   53 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/65.0%/33.1Hz_62.1Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   51 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/70.0%/33.1Hz_62.1Hz_0Hz.parquet
    Excited neurons: 48


WARNING    /home/se/anaconda3/envs/flywire/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]


    Elapsed time:   52 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/75.0%/33.1Hz_62.1Hz_0Hz.parquet
    Excited neurons: 48


WARNING    /home/se/anaconda3/envs/flywire/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]


    Elapsed time:   54 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/80.0%/33.1Hz_62.1Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   50 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/85.0%/33.1Hz_62.1Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   52 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/90.0%/33.1Hz_62.1Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   53 s
>>> Experiment:     33.1Hz_62.1Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin

WARNING    /home/se/anaconda3/envs/flywire/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
 [py.warnings]


    Elapsed time:   51 s
>>> Experiment:     69.5Hz_76.2Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/50.0%/69.5Hz_76.2Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   52 s
>>> Experiment:     69.5Hz_76.2Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/55.0%/69.5Hz_76.2Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   52 s
>>> Experiment:     69.5Hz_76.2Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/60.0%/69.5Hz_76.2Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   52 s
>>> Experiment:     69.5Hz_76.2Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin

    Elapsed time:   51 s
>>> Experiment:     88.4Hz_82.5Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/95.0%/88.4Hz_82.5Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   51 s
>>> Experiment:     127.2Hz_96.7Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/0.0%/127.2Hz_96.7Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   49 s
>>> Experiment:     127.2Hz_96.7Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/5.0%/127.2Hz_96.7Hz_0Hz.parquet
    Excited neurons: 48
    Elapsed time:   51 s
>>> Experiment:     127.2Hz_96.7Hz_0Hz
    Output file:    /volume_4/research/seongbong/flywire/geos

In [16]:
for ff1,ff2 in firing[1:]:
    for p in np.arange(0,1,0.05):
        percent = f'{np.round(p*100,2)}%'

        params['w_syn'] = 0.275*mV
        params['n_run'] = 100
        config = {
            'path_res'  : f'{os.getcwd()}/result/{percent}',                              # directory to store results
            'path_comp' : REPO_ROOT / "data" /'Completeness_783.csv',    
            'path_con'  : f'{os.getcwd()}/data/synapse_edges_rm_labial_{percent}.parquet',    # connectivity data
            'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
        }

        if percent not in os.listdir(f'{os.getcwd()}/result'):
            os.mkdir(config['path_res'])
        if 'input_spike_pattern' not in os.listdir(f'{os.getcwd()}/result/{percent}'):
            os.mkdir(f"{config['path_res']}/input_spike_pattern")
        if 'params' not in os.listdir(f'{config["path_res"]}'):
            os.mkdir(f'{config["path_res"]}/params')

        pickle.dump(params,open(f'{config["path_res"]}/params/params.pkl','wb'))


        neu_exc = atgrn_c2g['a6']+atgrn_c2g['a7']

        neu_add = [TPN1,interest_neurons]
        except_output_ids = [atgrn_c2g['a6']+atgrn_c2g['a7']+TPN1]
        except_output_wsyn = [0.825*mV]
        except_input_ids = [interest_neurons]
        except_input_wsyn = [0*mV]
        neu_slnc = []

        spike_path = REPO_ROOT/f'figure4/concentration_mapped_simulation/result_atGRN&TPN1/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'        d = pickle.load(open(spike_path,'rb'))

        av1a1_spike_path = f'../../geosmin_effect_activation_of_same_set_of_interneurons/inhibitory_spike_data/{0}Hz_{0}Hz_170Hz.pkl'
        df_av1a1 = pickle.load(open(av1a1_spike_path,'rb'))
        
        spike_df = [{1:d[i][1],2:d[i][2],3:df_av1a1[i][3]} for i in range(params['n_run'])]

        with open(f'{config["path_res"]}/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl','wb') as f:
            pickle.dump(spike_df,f)

        spike_path = f'{config["path_res"]}/input_spike_pattern/{ff1}Hz_{ff2}Hz_170Hz.pkl'
        predetermined_input_or_not = [1,1,1]
        params['r_poi1'] = ff1 * Hz
        params['r_poi2'] = ff2 * Hz
        params['r_poi3'] = 170 * Hz
        run_exp(exp_name=f'{ff1}Hz_{ff2}Hz_170Hz', neu_exc=neu_exc,neu_exc_add=neu_add,Except_output_Ids=except_output_ids,Except_output_w_syn=except_output_wsyn,Except_input_Ids=except_input_ids,Except_input_w_syn=except_input_wsyn,neu_slnc=neu_slnc,params=params,predetermined_input_or_not=predetermined_input_or_not,spike_path=spike_path, **config)

>>> Experiment:     33.1Hz_62.1Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/0.0%/33.1Hz_62.1Hz_170Hz.parquet
    Excited neurons: 48
    Elapsed time:   61 s
>>> Experiment:     33.1Hz_62.1Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/5.0%/33.1Hz_62.1Hz_170Hz.parquet
    Excited neurons: 48
    Elapsed time:   53 s
>>> Experiment:     33.1Hz_62.1Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/10.0%/33.1Hz_62.1Hz_170Hz.parquet
    Excited neurons: 48
    Elapsed time:   50 s
>>> Experiment:     33.1Hz_62.1Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_vers

    Elapsed time:   51 s
>>> Experiment:     69.5Hz_76.2Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/40.0%/69.5Hz_76.2Hz_170Hz.parquet
    Excited neurons: 48
    Elapsed time:   50 s
>>> Experiment:     69.5Hz_76.2Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/45.0%/69.5Hz_76.2Hz_170Hz.parquet
    Excited neurons: 48
    Elapsed time:   51 s
>>> Experiment:     69.5Hz_76.2Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/50.0%/69.5Hz_76.2Hz_170Hz.parquet
    Excited neurons: 48
    Elapsed time:   50 s
>>> Experiment:     69.5Hz_76.2Hz_170Hz
    Output file:    /volume_4/research/seongbong/f

    Elapsed time:   52 s
>>> Experiment:     88.4Hz_82.5Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/80.0%/88.4Hz_82.5Hz_170Hz.parquet
    Excited neurons: 48
    Elapsed time:   52 s
>>> Experiment:     88.4Hz_82.5Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/85.0%/88.4Hz_82.5Hz_170Hz.parquet
    Excited neurons: 48
    Elapsed time:   52 s
>>> Experiment:     88.4Hz_82.5Hz_170Hz
    Output file:    /volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/tarsal/simulation/result/90.0%/88.4Hz_82.5Hz_170Hz.parquet
    Excited neurons: 48
    Elapsed time:   51 s
>>> Experiment:     88.4Hz_82.5Hz_170Hz
    Output file:    /volume_4/research/seongbong/f